# Conservative vs. point-interpolation remapping of EERIE precipitation biases

**Does area-conservative remapping change the precipitation biases we report?**

Point-interpolation schemes (`nearest`, `linear`) do not preserve an area integral: a target cell
takes the value of one source point, or a distance-weighted blend, rather than the area-weighted
mean of the source cells it covers. Coarsening precipitation that way biases the domain total *and*
reports point values as box means. `feather` therefore remaps flux and precipitation fields
**area-conservatively** (`nereus.conservative_fluxes`).

This notebook compares two completed `precipitation_mswep` runs of the same 10-member EERIE
ensemble against MSWEP v2.8, 1980–2014, on a common 0.25° grid.

| run | `pr` remapping |
|-----|----------------|
| `output_precip_linear` | `linear` (Delaunay triangulation) |
| `output_eerie_10_mems_cmip6` | `conservative` (area overlap) |

> **Read the baseline carefully.** `output_precip_linear` is a *linear-everywhere* run. It is
> neither the historical default (model/obs regridding was **nearest**) nor current behaviour
> (**conservative** for fluxes, **nearest** for non-flux). Treat it as a contrast, not as "before".

> **Cost.** Everything here reads only the saved PNGs and JSON sidecars, so it is login-node safe —
> no model data is opened and nothing is regridded.

## 1. Configuration

In [ ]:
# Keep BLAS from spawning a thread per core (avoids RLIMIT_NPROC errors on DKRZ).
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

Image.MAX_IMAGE_PIXELS = None   # the combined panels are large
plt.rcParams["figure.dpi"] = 110

ROOT = Path("/work/bm1344/AWI/EERIE")
RUNS = {
    "linear":       ROOT / "output_precip_linear"        / "figures" / "precipitation_mswep",
    "conservative": ROOT / "output_eerie_10_mems_cmip6"  / "figures" / "precipitation_mswep",
}
UNITS = "mm/day"

for name, d in RUNS.items():
    n = len(list(d.glob("*.png"))) if d.is_dir() else 0
    print(f"{name:13} {n:2d} figures  {d}")

## 2. Are the two runs comparable?

Before comparing numbers, check that both runs actually produced the same figures over the same
models and period. A missing figure set silently invalidates the comparison — an earlier attempt at
this comparison failed exactly that way, with one run missing every bias map.

In [ ]:
def figure_ids(d):
    return {p.stem for p in d.glob("*.png")}

ids = {k: figure_ids(v) for k, v in RUNS.items()}
common = sorted(set.intersection(*ids.values()))
only = {k: sorted(v - set(common)) for k, v in ids.items()}

print(f"figures in both runs : {len(common)}")
for k, v in only.items():
    print(f"only in {k:13}: {v if v else 'none'}")

meta = {k: json.load(open(RUNS[k] / "pr_annual_bias_combined.json")) for k in RUNS}
for k, m in meta.items():
    print(f"\n{k}: period={m['period']}  obs={m['obs_dataset']}  n_models={len(m['models'])}")
assert meta["linear"]["models"] == meta["conservative"]["models"], "model lists differ"
assert meta["linear"]["period"] == meta["conservative"]["period"], "periods differ"
print("\nSame models, same period -> comparable.")

## 3. Per-model bias statistics

The annual `*_bias_combined.json` sidecars carry per-model statistics against MSWEP. (Seasonal
figures record only the benchmark MMM, so annual is the substantive comparison.)

Each metric has its own notion of "better":

| metric | better when |
|--------|-------------|
| `rmse` | lower |
| `pattern_correlation` | higher |
| `std_ratio` | closer to 1 |
| `*_mean_bias` | smaller in absolute value |

In [ ]:
METRICS = ["global_mean_bias", "rmse", "pattern_correlation",
           "std_ratio", "tropical_mean_bias", "extratropical_mean_bias"]

def stats_frame(period="annual"):
    rows = []
    s = {k: json.load(open(RUNS[k] / f"pr_{period}_bias_combined.json"))["summary_statistics"]
         for k in RUNS}
    for model in s["linear"]:
        if model not in s["conservative"]:
            continue
        for m in METRICS:
            a, b = s["linear"][model].get(m), s["conservative"][model].get(m)
            if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
                continue
            rows.append({"model": model, "metric": m, "linear": a, "conservative": b,
                         "delta": b - a})
    return pd.DataFrame(rows)

def is_better(metric, a, b):
    if metric == "rmse":                return b < a
    if metric == "pattern_correlation": return b > a
    if metric == "std_ratio":           return abs(b - 1) < abs(a - 1)
    return abs(b) < abs(a)

df = stats_frame("annual")
df["better"] = [is_better(r.metric, r.linear, r.conservative) for r in df.itertuples()]
df.pivot(index="model", columns="metric", values="delta").round(6)

### 3.1 Which direction does each metric move?

In [ ]:
summary = (df.groupby("metric")
             .agg(models=("model", "size"),
                  improved=("better", "sum"),
                  median_delta=("delta", "median"))
             .assign(share=lambda d: (100 * d.improved / d.models).round(0))
             .sort_values("share", ascending=False))
summary

A metric that improves for *every* model is a systematic effect of the remapping, not noise.
One that splits roughly evenly is not moving in any meaningful direction.

### 3.2 RMSE and pattern correlation, per model

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, unit in [(axes[0], "rmse", UNITS),
                         (axes[1], "pattern_correlation", "")]:
    sub = df[df.metric == metric].set_index("model")
    y = np.arange(len(sub))
    ax.barh(y - 0.2, sub["linear"], 0.4, label="linear", color="#b0b0b0")
    ax.barh(y + 0.2, sub["conservative"], 0.4, label="conservative", color="#1f77b4")
    ax.set_yticks(y); ax.set_yticklabels(sub.index, fontsize=8)
    ax.set_xlabel(f"{metric} [{unit}]" if unit else metric)
    ax.invert_yaxis(); ax.legend(fontsize=8)
    if metric == "pattern_correlation":
        lo = float(sub[["linear", "conservative"]].min().min())
        ax.set_xlim(min(0.7, lo - 0.02), 1.0)
fig.suptitle("Annual pr bias vs MSWEP — linear vs conservative", fontweight="bold")
fig.tight_layout()

The absolute bars look near-identical, which is the honest headline: the *sign* of the change
is consistent but its *size* is small. The plot below shows the change alone.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for metric, colour, scale, label in [
        ("rmse", "#d62728", 100, "RMSE change [%] (negative = better)"),
        ("pattern_correlation", "#2ca02c", 1000, "corr change x1000 (positive = better)")]:
    sub = df[df.metric == metric].set_index("model")
    rel = (sub["delta"] / sub["linear"] * scale) if metric == "rmse" else sub["delta"] * scale
    ax.plot(rel.values, np.arange(len(sub)), "o-", color=colour, label=label)
ax.axvline(0, color="k", lw=0.8)
ax.set_yticks(np.arange(len(sub))); ax.set_yticklabels(sub.index, fontsize=8)
ax.invert_yaxis(); ax.legend(fontsize=8); ax.set_xlabel("change (conservative - linear)")
ax.set_title("Effect of conservative remapping, per model", fontweight="bold")
fig.tight_layout()

**Why is the effect small?** The model side is 0.25° → 0.25°, essentially the identity. The
step that genuinely aggregates is **MSWEP 0.1° → 0.25°**, and that changes the *reference* every
model is scored against — so the models move together rather than apart.

## 4. Rendering artefacts

`linear` interpolation triangulates, and Delaunay cannot close the 0°/360° longitude seam, leaving a
one-cell NaN column that renders as a white line down the centre of each panel. Conservative
remapping uses polygon overlap and has no seam to fail at.

The detector below looks for a narrow, near-white column whose neighbours are *not* white — i.e.
page background showing through the map, rather than a genuine zero in the colormap.

In [ ]:
def find_stripes(png, row_lo=0.12, row_hi=0.20, white=250, min_frac=0.85, nb_frac=0.40):
    """Column indices of narrow page-white stripes inside the map area."""
    im = np.asarray(Image.open(png).convert("RGB")).astype(int)
    H, W, _ = im.shape
    band = im[int(row_lo * H):int(row_hi * H), :, :]
    frac = ((band > white).all(axis=2)).mean(axis=0)
    return [c for c in range(6, frac.size - 6)
            if frac[c] > min_frac and 0.5 * (frac[c - 5] + frac[c + 5]) < nb_frac], W

rows = []
for fid in ["pr_annual_bias_combined", "pr_annual_relative_bias",
            "pr_djf_bias_combined", "pr_jja_bias_combined"]:
    rec = {"figure": fid}
    for run, d in RUNS.items():
        cols, W = find_stripes(d / f"{fid}.png")
        rec[run] = ", ".join(f"{100*c/W:.0f}%" for c in cols) if cols else "none"
    rows.append(rec)
pd.DataFrame(rows).set_index("figure")

Stripe positions cluster at the centre of each panel column (~17 %, ~50 %, ~83 % across a
three-column figure) — that is longitude 0 in each Robinson panel.

### 4.1 Zoom on the seam

In [ ]:
fid = "pr_annual_bias_combined"
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (run, d) in zip(axes, RUNS.items()):
    im = np.asarray(Image.open(d / f"{fid}.png").convert("RGB"))
    H, W, _ = im.shape
    # a narrow window around the centre of the first bias panel
    c = int(0.50 * W)
    ax.imshow(im[int(0.10 * H):int(0.22 * H), c - 60:c + 60])
    ax.set_title(f"{run}  (panel centre = lon 0)", fontsize=10)
    ax.axis("off")
fig.suptitle("Prime-meridian seam, " + fid, fontweight="bold")
fig.tight_layout()

## 5. How much of the map actually changes?

In [ ]:
rows = []
for fid in sorted(common):
    a = np.asarray(Image.open(RUNS["linear"] / f"{fid}.png").convert("RGB")).astype(int)
    b = np.asarray(Image.open(RUNS["conservative"] / f"{fid}.png").convert("RGB")).astype(int)
    if a.shape != b.shape:
        rows.append({"figure": fid, "pixels_differing_%": np.nan, "note": "shape mismatch"})
        continue
    d = (np.abs(a - b).max(axis=2) > 8).mean() * 100
    rows.append({"figure": fid, "pixels_differing_%": round(d, 2), "note": ""})
pd.DataFrame(rows).set_index("figure").sort_values("pixels_differing_%", ascending=False)

Figures with **0 %** differing pixels are byte-identical — these are the diagnostics that never
touch the common grid (time series, seasonal cycle, zonal mean, intensity PDF), so the remapping
method cannot affect them. That they are unchanged is a useful control: it confirms the differences
elsewhere come from the regridding and not from run-to-run variation.

### 5.1 Side-by-side figures

In [ ]:
def show_pair(fid, crop=None):
    fig, axes = plt.subplots(2, 1, figsize=(14, 13))
    for ax, (run, d) in zip(axes, RUNS.items()):
        im = np.asarray(Image.open(d / f"{fid}.png").convert("RGB"))
        ax.imshow(im[crop] if crop else im)
        ax.set_title(f"{fid} — {run}", fontsize=11, fontweight="bold")
        ax.axis("off")
    fig.tight_layout()

show_pair("pr_annual_bias_combined")

In [ ]:
show_pair("pr_annual_relative_bias")

## 6. Summary

Fill in from the tables above; the pattern observed on the 1980–2014 EERIE run was:

- **RMSE improved for every model** and **pattern correlation improved for nearly all** — small
  (~0.5–1 %) but entirely systematic, which is the signature of a better-conserved *reference*
  field rather than changed model fields.
- **Global and tropical mean bias barely moved**, and marginally increased in magnitude. Conservative
  remapping is not a uniform improvement to every statistic, and reporting it as one would be wrong.
- **`std_ratio` was unchanged** in any meaningful sense.
- **The prime-meridian seam disappeared**, because conservative remapping does not triangulate.

The limiting factor on how much this can matter is that the models are already on the target grid.
Conservative remapping earns its cost where a genuine coarsening happens — here that is MSWEP
0.1° → 0.25°, and in the DestinE configs it would be the ~5 km native grids, which currently exceed
`nereus.conservative_max_points` and fall back to point interpolation.